In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="softmax_gap_mean_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [9]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [10]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()} if hasattr(model.config.id2label, "items") else dict(enumerate(model.config.id2label))
label_text = {i: str(v).lower() for i, v in id2label.items()}
entailment_id = next(i for i, v in label_text.items() if "entail" in v)
contradiction_id = next(i for i, v in label_text.items() if "contrad" in v)

print("model_name:", model_name)
print("id2label:", id2label)
print("entailment_id:", entailment_id, "contradiction_id:", contradiction_id)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model_name: typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
entailment_id: 0 contradiction_id: 2


In [11]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [12]:
batch_size = 64
preds = []
gap_12_all = []
gap_21_all = []
avg_gap_all = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc_12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc_21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        enc_12 = {k: v.to(device) for k, v in enc_12.items()}
        enc_21 = {k: v.to(device) for k, v in enc_21.items()}

        logits_12 = model(**enc_12).logits
        logits_21 = model(**enc_21).logits

        probs_12 = torch.softmax(logits_12, dim=-1)
        probs_21 = torch.softmax(logits_21, dim=-1)

        gap_12 = probs_12[:, entailment_id] - probs_12[:, contradiction_id]
        gap_21 = probs_21[:, entailment_id] - probs_21[:, contradiction_id]
        avg_gap = (gap_12 + gap_21) / 2.0

        batch_preds = (avg_gap > 0).long().cpu().numpy()

        preds.extend(batch_preds.tolist())
        gap_12_all.extend(gap_12.cpu().numpy().tolist())
        gap_21_all.extend(gap_21.cpu().numpy().tolist())
        avg_gap_all.extend(avg_gap.cpu().numpy().tolist())

y_pred = np.array(preds)
gap_12_all = np.array(gap_12_all)
gap_21_all = np.array(gap_21_all)
avg_gap_all = np.array(avg_gap_all)

print("done")

  0%|          | 0/7 [00:00<?, ?it/s]

done


In [ ]:
vault.create_record_list("mrpc_mean_gap_prediction", column_names=["prediction", "gap_12", "gap_21"])

for i in range(len(y_pred)):
    vault.append_record("mrpc_mean_gap_prediction", 
                        {
                            "prediction": y_pred[i],
                            "gap_12": float(gap_12_all[i]),
                            "gap_21": float(gap_21_all[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Per-example prediction dataset for the GLUE MRPC validation split, generated by running the sentence pairs through the typeform/distilbert-base-uncased-mnli model in both input orders. Each record corresponds to one MRPC validation example and is linked to the matching row in glue_mrpc_validation. The dataset has three fields: prediction (binary paraphrase decision, where 1 indicates paraphrase and 0 indicates not paraphrase), gap_12 (entailment probability minus contradiction probability for sentence1\u2192sentence2), and gap_21 (the same probability gap for sentence2\u2192sentence1). In this workflow, it serves as the stored per-instance model output used to inspect directional confidence, compare predictions with ground-truth MRPC labels, and build the downstream summary metrics in softmax_gap_mean_mrpc_summary."
embedding = get_embeddings(description)
vault.create_description("mrpc_mean_gap_prediction", description, embedding)

properties = {"task": "paraphrase detection", "data_type": "model predictions", "prediction_type": "binary classification", "source": "glue/mrpc", "derived_from": "glue_mrpc_validation", "split": "validation", "size": "408", "input_type": "sentence pair", "model": "typeform/distilbert-base-uncased-mnli", "method": "mean softmax entailment-contradiction gap over both sentence orders", "columns": "prediction,gap_12,gap_21", "label_space": "0=not_paraphrase,1=paraphrase"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_mean_gap_prediction", cat, embedding, prop)

In [13]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.7181372549019608, 'f1': 0.7905282331511839}
                precision    recall  f1-score   support

not_paraphrase       0.55      0.59      0.57       129
    paraphrase       0.80      0.78      0.79       279

      accuracy                           0.72       408
     macro avg       0.68      0.68      0.68       408
  weighted avg       0.72      0.72      0.72       408



In [14]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("gap_12:", float(gap_12_all[i]))
    print("gap_21:", float(gap_21_all[i]))
    print("avg_gap:", float(avg_gap_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
gap_12: 0.9963528513908386
gap_21: 0.8122128844261169
avg_gap: 0.9042828679084778
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
gap_12: -0.9993698596954346
gap_21: -0.000674131908454001
avg_gap: -0.5000219941139221
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
gap_12: -0.00016976663027890027
gap_21: 0.9945570826530457
avg_gap: 0.49719

In [15]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("gap_12:", float(gap_12_all[i]))
    print("gap_21:", float(gap_21_all[i]))
    print("avg_gap:", float(avg_gap_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

num_errors: 115
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
gap_12: -0.00016976663027890027
gap_21: 0.9945570826530457
avg_gap: 0.49719366431236267
true: 0 pred: 1
idx: 4
sentence1: No dates have been set for the civil or the criminal trial .
sentence2: No dates have been set for the criminal or civil cases , but Shanley has pleaded not guilty .
gap_12: -0.0002203288022428751
gap_21: 0.9956680536270142
avg_gap: 0.49772384762763977
true: 0 pred: 1
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
gap_12: -0.00019033583521377295
gap_21: -0.0004739669

In [16]:
vault.create_record_list("softmax_gap_mean_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("softmax_gap_mean_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_mean_gap_prediction": [0, len(ds)]
                    })

summary

description = "Summary dataset for the MRPC validation experiment using an MNLI sequence-classification model and the mean softmax-gap decision rule. It contains a single aggregate record with three fields: accuracy (float), f1 (float), and classification_report (string). These metrics are computed by comparing ground-truth MRPC labels from glue_mrpc_validation against predictions in mrpc_mean_gap_prediction, where each prediction is based on the average of the entailment-minus-contradiction probability gap for both sentence orders (sentence1, sentence2) and (sentence2, sentence1). In this workflow, this dataset serves as the experiment-level evaluation output summarizing model performance over the full validation set."
embedding = get_embeddings(description)
vault.create_description("softmax_gap_mean_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "benchmark": "GLUE", "source": "glue_mrpc_validation", "subset": "MRPC", "split": "validation", "size": "408", "model": "typeform/distilbert-base-uncased-mnli", "inference_method": "mean softmax entailment-contradiction gap over both sentence orders", "metrics": "accuracy,f1,classification_report", "input_prediction_dataset": "mrpc_mean_gap_prediction", "process_name": "softmax_gap_mean_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("softmax_gap_mean_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.7181372549019608,
 'f1': 0.7905282331511839,
 'method': 'mean_bidirectional_softmax_gap'}

In [ ]:
description = "This notebook runs a paraphrase detection experiment on the GLUE MRPC validation set using the typeform/distilbert-base-uncased-mnli sequence classification model in an NLI-style formulation. For each sentence pair, it scores both input orders (sentence1, sentence2) and (sentence2, sentence1), computes the softmax probability gap between entailment and contradiction for each order, averages the two gaps, and predicts paraphrase when the mean gap is greater than zero. The workflow loads the MRPC validation examples from TableVault, performs batched inference with Hugging Face Transformers and PyTorch, and stores per-example predictions and gap scores back into TableVault with lineage to the source records. It then evaluates the method with accuracy, F1, and a classification report, inspects sample predictions and errors, and saves an experiment summary plus descriptive metadata and embeddings for the prediction table, summary table, and overall notebook process." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("softmax_gap_mean_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "dataset": "glue/mrpc", "dataset_split": "validation", "model": "typeform/distilbert-base-uncased-mnli", "model_family": "distilbert", "inference_type": "zero-shot NLI", "prediction_method": "mean entailment-contradiction softmax gap over sentence order swaps", "input_format": "sentence pair classification", "framework": "transformers, pytorch", "metrics": "accuracy, f1-score, classification report", "outputs": "per-example predictions and summary metrics", "tracking": "tablevault", "vector_embedding_model": "text-embedding-3-large", "llm_provider": "openai"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("softmax_gap_mean_mrpc", cat, embedding, prop)